In [0]:
# ---------------------------------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 4.2_fato_ceap_despesas
# LOCAL: 3_gold/src/4_gastos_ceap/
# OBJETIVO: Gerar fato de gastos com fornecedores usando a tabela gerada pelo script 1_bronze/src/4_gastos_ceap/4.1_ceap_despesas_incremental
# ENTREGA: Tabela fato_despesas com dimensões: deputado, fornecedor, categoria, mês
# ---------------------------------------------------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, to_date

# Carregar dados
df_bronze = spark.table("bronze_ceap_despesas")
df_deputados = spark.table("gold_atlas_frentes_parlamentares") \
    .select(col("id_deputado").cast("long"), "nome_deputado", "uf", "partido").distinct()

# Construção da Fato
df_fato = df_bronze.select(
    col("codDocumento").alias("id_despesa"),
    col("idDeputado_fix").cast("long").alias("id_deputado"), 
    col("tipoDespesa").alias("categoria"),
    col("dataDocumento").alias("data_referencia"), 
    col("valorDocumento").alias("valor_gasto"),
    col("cnpjCpfFornecedor").alias("cnpj_fornecedor"),
    col("nomeFornecedor").alias("nome_fornecedor")
).join(df_deputados, "id_deputado", "inner")

# Converter data para formato Spark e tratar tipos
df_fato = df_fato.withColumn("data_referencia", to_date(col("data_referencia")))

# Salvar na Gold
df_fato.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fato_ceap_despesas")

display(df_fato)